# Creating Python Code Files for SAS Intelligent Decisioning

This notebook demonstrates how to use the `CodeFile` class to upload Python code files formatted for SAS Intelligent Decisioning.

## Overview


SAS Intelligent Decisioning (ID) requires Python code files to follow a specific format for detailed specifications on Python code file format requirements for SAS Intelligent Decisioning, see the [Rules For Developing Python Code Files](https://go.documentation.sas.com/doc/en/edmcdc/v_063/edmug/n04vfc1flrz8jsn1o5jblnbgx6i3.htm#n0jrohir6wzvd0n11omfautducm3) documentation.

A basic overview:
- An `execute` function is required
- An `Output:` docstring listing output variables as first line in the execute function
- A `DependentPackages:` docstring listing required packages at the top of the file including all non built-in packages needed
- Must return standard Python data types


The `CodeFile` class validates and uploads properly formatted Python code to SAS Viya.

## Prerequisites

- A SAS Viya environment with Intelligent Decisioning
- Appropriate permissions to create files in the target folder
- sasctl package installed
- Python code already formatted according to ID requirements

## Setup: Connect to SAS Viya

In [ ]:
from sasctl import Session
from sasctl.pzmm import CodeFile
from sasctl.services import folders as folder_service


# Replace with your SAS Viya connection details
HOST = 'your-viya-host.com'
USERNAME = 'your-username'
PASSWORD = 'your-password'

# Create a session
sess = Session(HOST, USERNAME, PASSWORD, verify_ssl=False)
print(f"Connected to {HOST}")

try:
    folder_service.create_folder('ID_python_files', "/Public")
except Exception as error:
    print(f"Folder already exists. {error}")

/Users/sababa/Desktop/repos/python-sasctl/venv3.11/lib/python3.11/site-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'base.ingress-nginx.sababa-dq1-m1.modelmanager.sashq-d.openstack.sas.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


Connected to https://base.ingress-nginx.sababa-dq1-m1.modelmanager.sashq-d.openstack.sas.com/
Folder already exists. HTTP Error 409: {"version":2,"httpStatusCode":409,"errorCode":11552,"message":"An item named \"ID_python_files\" of type \"Folder\" already exists in the folder \"Public\".","details":["Existing member: ","/folders/folders/3789dfcd-a5a6-4836-85d9-beb5f812baf8","Suggestion: ID_python_files (1)","path: /folders/folders","correlator: 58004f45-b0cb-4737-adb1-1edbfba2040c"]}


## Example 1: Simple Code File

Let's start with a simple example that performs a basic calculation.

In [ ]:
# Define properly formatted ID Python code
simple_code = """
def execute(input_value):
    '''Output: score, category'''
    # Calculate a simple score
    score = input_value * 2 + 10
    category = 'High' if score > 50 else 'Low'
    return score, category
"""

# Upload the code file to Viya
file_obj = CodeFile.write_id_code_file(
    code=simple_code,
    file_name='simple_calculator.py',
    folder='/Public/ID_python_files'
)

print(f"File uploaded successfully!")
print(f"File ID: {file_obj.id}")
print(f"File Name: {file_obj.name}")

File uploaded successfully!
File ID: 5169bfbe-4ba4-4998-b9c3-24228add86a7
File Name: simple_calculator


## Example 2: Code File with API Call

This example shows how to create a code file that makes an API call to retrieve data.

In [11]:
api_code = """
'''DependentPackages: requests'''
def execute(customer_id):
    '''Output: risk_score, status'''
    import requests
    import json

    # Make an API call
    url = f"https://api.example.com/data?id={customer_id}"
    response = requests.get(url)

    if response.status_code == 200:
        data = response.json()
        risk_score = data.get('risk_score', 0)
        status = 'Success'
    else:
        risk_score = -1
        status = 'Failed'
    
    return risk_score, status
"""

# Upload the code file
file_obj = CodeFile.write_id_code_file(
    code=api_code,
    file_name='risk_score_api.py',
    folder='/Public/ID_python_files'
)

print(f"File uploaded successfully!")
print(f"File ID: {file_obj.id}")
print(f"File Name: {file_obj.name}")

File uploaded successfully!
File ID: 7484adcd-3121-4e13-b208-c7b1ef51e444
File Name: risk_score_api


## Example 3: Code with Multiple Dependencies

Specify multiple packages in the DependentPackages docstring.

In [12]:
data_processing_code = """
'''DependentPackages: pandas, numpy'''
def execute(value1, value2, value3, threshold):
    '''Output: mean_value, std_value, result'''
    import pandas as pd
    import numpy as np

    # Create a simple dataframe
    data = pd.DataFrame({
        'values': [value1, value2, value3]
    })

    # Calculate statistics
    mean_value = float(np.mean(data['values']))
    std_value = float(np.std(data['values']))
    result = 'Pass' if mean_value > threshold else 'Fail'
    
    return mean_value, std_value, result
"""

# Upload the code file
file_obj = CodeFile.write_id_code_file(
    code=data_processing_code,
    file_name='data_processor.py',
    folder='/Public/ID_python_files'
)

print(f"File uploaded successfully: {file_obj.name}")

File uploaded successfully: data_processor


## Example 4: Reading Code from a File

You can also read Python code from an existing file.

In [ ]:
from pathlib import Path

# Create a properly formatted Python file
temp_code_file = Path('temp_code.py')
temp_code_file.write_text("""
def execute(income, assets, debt):
    '''Output: credit_score, decision, confidence'''
    # Business logic for credit decision
    credit_score = income * 0.3 + assets * 0.2 - debt * 0.5
    decision = 'Approved' if credit_score > 650 else 'Denied'
    confidence = min(credit_score / 850, 1.0)
    
    return credit_score, decision, confidence
""")

# Upload code from file (pass Path object)
file_obj = CodeFile.write_id_code_file(
    code=temp_code_file,
    file_name='credit_decision.py',
    folder='/Public/ID_python_files'
)

# Clean up
temp_code_file.unlink()

print(f"Uploaded code from file: {file_obj.name}")

Uploaded code from file: credit_decision


## Example 5: Code File with No Parameters

You can also create code files that don't require input parameters.

In [ ]:
from sasctl.services import files as file_service
from sasctl.services import folders as folder_service

config_code = """
def execute():
    '''Output: current_date, environment, version'''
    import datetime

    # Get current configuration
    current_date = datetime.datetime.now().strftime('%Y-%m-%d')
    environment = 'production'
    version = '1.0.0'
    
    return current_date, environment, version
"""

# Check if file already exists and delete it
# WARNING: Deleting files may result in loss of important data or configurations.
# Ensure you have backups or that the file can be safely removed before proceeding.

file_name = 'config_info.py'
folder_path = '/Public/ID_python_files'

try:
        folder_obj = folder_service.get_folder(folder_path)

        file_filter = f"and(eq(name, '{file_name}'), eq(contentType, 'file'))"
        existing_file = folder_service.get(
            f"/folders/{folder_obj.id}/members",
            params={"filter": file_filter}
        )
        if len(existing_file) > 0:
            print(f"WARNING: About to delete existing file: {file_name}")
            print("This may result in loss of sensitive data or configurations.")

            file_service.delete_file({"id": existing_file['uri'].split('/')[-1]})
            print(f"Deleted existing file: {file_name}")
except Exception as e:
    print(f"No existing file found: {file_name} {e}")


file_obj = CodeFile.write_id_code_file(
    code=config_code,
    file_name=file_name,
    folder=folder_path
)

print(f"Configuration code file created: {file_name}")

This may result in loss of sensitive data or configurations.
Deleted existing file: config_info.py
Configuration code file created: config_info


## Example 6: Disable Validation

You can skip pre-upload validation **Note:** The file will still be uploaded even if it has formatting errors - those errors will appear later when you try to use the file in a decision. You can view the codeFile in Intelligent Decisioning and validate it to check.

In [ ]:
fast_code = """
def execute(input_a, input_b):
    '''Output: result'''
    result = input_a + input_b
    return result
"""

# Skip pre-upload validation for faster upload
# File will still be created even if there are formatting errors
file_obj = CodeFile.write_id_code_file(
    code=fast_code,
    file_name='fast_calculator.py',
    folder='/Public/ID_python_files',
    validate_code=False  # Skip pre-upload validation
)

print(f"File uploaded without pre-validation: {file_obj.name}")
print("Warning: If there are formatting errors, they will appear when you use the file in a decision.")

File uploaded without pre-validation: fast_calculator


## Clean Up

Close the SAS Viya session when finished.

In [16]:
# Close the session
sess.close()
print("Session closed")

Session closed


## Additional Resources

- [SAS Intelligent Decisioning Documentation](https://go.documentation.sas.com/doc/en/edmcdc/v_063/edmug/n04vfc1flrz8jsn1o5jblnbgx6i3.htm)
- [Rules For Developing Python Code Files](https://go.documentation.sas.com/doc/en/edmcdc/v_063/edmug/n04vfc1flrz8jsn1o5jblnbgx6i3.htm#n0jrohir6wzvd0n11omfautducm3)
- [python-sasctl Documentation](https://sassoftware.github.io/python-sasctl/)